# HFT Tadawul — CUDA Analytics Benchmark

Benchmark CPU vs GPU multi-symbol order book analytics on real NASDAQ ITCH data.

**Before running:** in Colab go to *Runtime → Change runtime type → GPU (T4)*.

What this notebook does:
1. Verifies a CUDA GPU is available (T4 free tier is fine).
2. Mounts Google Drive and looks for the ITCH file. Downloads if missing.
3. Clones (or updates) the `tadawul-hft-engine` repo.
4. Builds the CPU + CUDA benchmark with `nvcc`.
5. Runs the benchmark across symbol counts (1, 10, 100, 1000, all 8K+).
6. Plots speedup and scalability curves.

Output written to `results/gpu_benchmark.csv` and saved back to Drive.

## 1. Verify GPU

In [ ]:
!nvidia-smi

In [ ]:
!nvcc --version

## 2. Mount Drive + download ITCH file (cached)

If the file already exists in your Drive at `MyDrive/tadawul-hft/01302020.NASDAQ_ITCH50.gz`, this skips the download (saves ~5 GB and 5–15 min).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, subprocess

DRIVE_DIR  = '/content/drive/MyDrive/tadawul-hft'
ITCH_NAME  = '01302020.NASDAQ_ITCH50.gz'
ITCH_URL   = f'https://emi.nasdaq.com/ITCH/Nasdaq%20ITCH/{ITCH_NAME}'
ITCH_PATH  = os.path.join(DRIVE_DIR, ITCH_NAME)

os.makedirs(DRIVE_DIR, exist_ok=True)

if os.path.exists(ITCH_PATH):
    size_mb = os.path.getsize(ITCH_PATH) / (1024*1024)
    print(f'✓ ITCH file found in Drive ({size_mb:.1f} MB): {ITCH_PATH}')
else:
    print(f'Downloading {ITCH_URL} → {ITCH_PATH}')
    print('(~5 GB, takes 5–15 minutes; cached in Drive for next time)')
    !wget -q --show-progress -O "{ITCH_PATH}" "{ITCH_URL}"
    print(f'✓ Downloaded ({os.path.getsize(ITCH_PATH) / (1024*1024):.1f} MB)')

## 3. Clone the engine repo

Edit `REPO_URL` if you've forked the repo (e.g. push to your own GitHub account).

In [ ]:
REPO_URL = 'https://github.com/kamelth/tadawul-hft-engine.git'
REPO_DIR = '/content/tadawul-hft-engine'

if os.path.isdir(REPO_DIR):
    print(f'Updating existing clone at {REPO_DIR}')
    !cd "{REPO_DIR}" && git pull --rebase
else:
    !git clone --depth 1 "{REPO_URL}" "{REPO_DIR}"

# Symlink the ITCH file into the repo's data/ dir so paths match local dev
!mkdir -p "{REPO_DIR}/data" "{REPO_DIR}/results" "{REPO_DIR}/build/manual"
!ln -sf "{ITCH_PATH}" "{REPO_DIR}/data/{ITCH_NAME}"
!ls -la "{REPO_DIR}/data"

## 4. Install dependencies + build

We need `zlib` (already on Colab) for ITCH decompression.

In [ ]:
!apt-get install -qq -y zlib1g-dev

In [ ]:
%%bash -e
cd /content/tadawul-hft-engine

# Detect GPU compute capability (T4 = 75, V100 = 70, A100 = 80, etc.)
CC=$(nvidia-smi --query-gpu=compute_cap --format=csv,noheader | head -1 | tr -d '.')
echo "GPU compute capability: sm_${CC}"

# Build CUDA kernel object
nvcc -O3 -std=c++17 -arch=sm_${CC} \
  -Iinclude -Imodules/core/include \
  -c source/gpu/analytics_kernel.cu \
  -o build/manual/analytics_kernel.o

# Build benchmark binary (HAVE_CUDA + WITH_ITCH)
nvcc -O3 -std=c++17 -arch=sm_${CC} \
  -Iinclude -Imodules/core/include \
  -DHAVE_CUDA -DWITH_ITCH \
  -Xcompiler "-O3 -march=native" \
  source/gpu/benchmark.cpp build/manual/analytics_kernel.o \
  -lz \
  -o build/manual/gpu_benchmark

ls -la build/manual/gpu_benchmark

## 5. Smoke test (synthetic data, fast)

In [ ]:
%%bash
cd /content/tadawul-hft-engine
./build/manual/gpu_benchmark --data synthetic --symbols 1,10,100,1000,5000 --iters 500

## 6. Real ITCH benchmark

5M ITCH messages → ~8,900 symbols. Sweep across 1, 10, 100, 1000, all symbols.

In [ ]:
%%bash
cd /content/tadawul-hft-engine
./build/manual/gpu_benchmark \
    --data itch \
    --itch-file data/01302020.NASDAQ_ITCH50.gz \
    --max-messages 5000000 \
    --symbols 1,10,100,1000,0 \
    --iters 1000

## 7. Plot speedup + scalability

In [ ]:
import csv, os
import matplotlib.pyplot as plt

csv_path = '/content/tadawul-hft-engine/results/gpu_benchmark.csv'
rows = []
with open(csv_path) as f:
    for row in csv.DictReader(f):
        rows.append({k: float(v) if k != 'num_symbols' else int(v) for k, v in row.items()})

ns      = [r['num_symbols']         for r in rows]
cpu_us  = [r['cpu_per_iter_us']     for r in rows]
gpu_kus = [r['gpu_kernel_per_iter_us'] for r in rows]
gpu_eus = [r['gpu_e2e_per_iter_us'] for r in rows]
sp_k    = [r['speedup_kernel']      for r in rows]
sp_e    = [r['speedup_e2e']         for r in rows]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(ns, cpu_us, 'o-', label='CPU (serial)', linewidth=2)
ax.plot(ns, gpu_kus, 's-', label='GPU (kernel only)', linewidth=2)
ax.plot(ns, gpu_eus, '^--', label='GPU (end-to-end incl. PCIe)', linewidth=2)
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Number of symbols')
ax.set_ylabel('Latency per analytics pass (µs)')
ax.set_title('Analytics Latency vs Symbol Count')
ax.grid(True, which='both', alpha=0.3)
ax.legend()

ax = axes[1]
ax.plot(ns, sp_k, 's-', label='Kernel-only speedup', linewidth=2)
ax.plot(ns, sp_e, '^--', label='End-to-end speedup', linewidth=2)
ax.axhline(y=1.0, color='r', linestyle=':', alpha=0.5, label='Break-even')
ax.set_xscale('log')
ax.set_xlabel('Number of symbols')
ax.set_ylabel('Speedup vs CPU (x)')
ax.set_title('GPU Speedup vs Symbol Count')
ax.grid(True, which='both', alpha=0.3)
ax.legend()

fig.suptitle('HFT Tadawul — CUDA Analytics Benchmark', fontsize=14)
fig.tight_layout()

out_png = '/content/tadawul-hft-engine/results/gpu_benchmark.png'
fig.savefig(out_png, dpi=120)
print(f'Saved {out_png}')
plt.show()

## 8. Save results back to Drive

In [ ]:
import shutil
DRIVE_RESULTS = os.path.join(DRIVE_DIR, 'results')
os.makedirs(DRIVE_RESULTS, exist_ok=True)

for fname in ['gpu_benchmark.csv', 'gpu_benchmark.png']:
    src = f'/content/tadawul-hft-engine/results/{fname}'
    if os.path.exists(src):
        dst = os.path.join(DRIVE_RESULTS, fname)
        shutil.copy(src, dst)
        print(f'Saved → {dst}')
    else:
        print(f'(skip) {src} not found')